
<img src= "image1.png">

Before building this chain we need to create 3 components:
- 1- retriever :  derived from the vector store 

- 2- prompt template : for combining the user question and retrieved context

- 3- LLM : a model to generate the response

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# charger le document .pdf (loading)
loader = PyPDFLoader("pdf_file_example.pdf")
documents = loader.load()

# Découper le document en morceaux (chunking)
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap= 200
)


chunks = splitter.split_documents(documents)


chunks

[Document(metadata={'source': 'pdf_file_example.pdf', 'page': 0}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi sit amet\ntortor quis risus auctor condimentum. Morbi in ullamcorper elit. Nulla iaculis tellus sit amet\nmauris tempus fringilla.\nMaecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.\n

In [ ]:
# Initialisation du modèle embedding, pour transformer le texte en vecteurs de nombres


from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    api_key = " "  
    model= "text-embedding-3-small"
)

In [ ]:
# On prend les morceaux de texte (chunks), 
# on les envoie à l'API d'OpenAI pour obtenir leurs vecteurs embeddings, 
# puis on stocke le tout dans une base de données vectorielle Chroma.



from langchain_chroma import Chroma 


vector_store = Chroma.from_documents(
    documents= chunks,
    embedding= embedding_model
)

### 1. Retriever

In [ ]:
# Configuration du retriever
# On fait la recherche par similarité (généralemebt avec la distance cosinus)
# On ne revoie que les 2 meilleurs morceaux avec k=2

retriever = vector_store.as_retriever(
    search_type = "similarity",  # the type of search in this case it is cosinesimilarity
    search_kwargs = {"k": 2} # how many chunks to retrieve when queried in this case it is 2
)

### 2. Prompt Template

In [2]:
# Then we create a prompt template
# On prépare le moule dans lequel la question et le contexte récupéré vont être injectés.


from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
                                            Use the following pieces of context to answer the question at the end.
                                            If you don't know the answer, say that you don't know.
                                            Context: {context}
                                            Question: {question}
                                            """)

prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n                                            Use the following pieces of context to answer the question at the end.\n                                            If you don't know the answer, say that you don't know.\n                                            Context: {context}\n                                            Question: {question}\n                                            "), additional_kwargs={})])

### 3. LLM

In [ ]:
# Then we define an OpenAI chat model to generate responses
# On initialise le modèle de chat avec une température = 0 pour que le modèle soit factuel (pas de créativité)

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model= "gpt-4o-mini", api_key= "...", temperature= 0)

# Build an LCEL retrieval chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


chain = (
    {"context": retriever, "question": RunnablePassthrough()} # Runnablepassthrough allow us to pass data without modifying it

    # then the retrieved "context" and user "question" are then passed into the prompt using the LCEL PIPE AND then to the LLM
    
    | prompt
    | llm
    
    # Then finally a string output parser is used to parse the model output as string
    
    | StrOutputParser()

)



# La chaine LCEL
# 1. On envoie une chaîne de caractères (la question).

# 2. Le retriever l'intercepte pour aller chercher le context, 
# tandis que RunnablePassthrough() laisse passer la question intacte.

# 3. Ces deux éléments remplissent les variables {context} et {question} du prompt.

# 4. Le prompt généré avec le moule est envoyé au llm.

# 5. StrOutputParser() extrait proprement la réponse sous forme de texte brut.

In [ ]:
# for example
result = chain.invoke("What are the key findings or results presented in the papaer?")

print(result)

## Meme code avec Ollama en local pour des modèles gratuits

<pre>
1.  Installation du logiicel ollama depuis ollama.com

2. Dans un terminal on va télécharger les deux modèles suivants:
    a. modèle de texte avec la commande:    ollama run llama3.2
    b. modèle embeddings avec la commande:  ollama pull nomic-embed-text

3. l'installation e langchain-ollama pour que Ollama puisse communiquer avec LangChain


In [12]:
# chargement du fichier pdf et split

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("Machine Learning Cheatsheet.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap= 200
)

chunks = splitter.split_documents(documents)

len(chunks)

13

In [13]:
# Initialisation du modèle embedding
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model= "nomic-embed-text"
)

embedding_model

OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [14]:
# envoyer les chunks au modèle embedding, et stocker les vecteurs dans chromadb

"""from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents= chunks,
    embedding= embedding_model

)""" # erreur kernel !!!!!!!!!!!!!!

from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [15]:
# configuration du retriever
retriever = vector_store.as_retriever(
    search_type = "similarity",  
    search_kwargs = {"k": 2}

)

In [16]:
# prompt template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""

                                            Use the following pieces of context to answer the question at the end.

                                            If you don't know the answer, say that you don't know.

                                            Context: {context}

                                            Question: {question}

                                            """)



prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n\n                                            Use the following pieces of context to answer the question at the end.\n\n                                            If you don't know the answer, say that you don't know.\n\n                                            Context: {context}\n\n                                            Question: {question}\n\n                                            "), additional_kwargs={})])

In [17]:

# llm
from langchain_ollama import ChatOllama
llm = ChatOllama(model= "llama3.2", 
                temperature= 0)




In [18]:
# La chaine LCEL

from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser


chain = (
    {"context": retriever, "question": RunnablePassthrough()} 
    | prompt
    | llm
        
    | StrOutputParser()

)



In [19]:
query = "What are the key findings or results presented in the paper?"
result = chain.invoke(query)

result

'Based on the provided context, it appears that the document is a machine learning cheatsheet. The key findings or results presented in the paper seem to be related to feature engineering and dimensionality reduction techniques.\n\nSpecifically, the section on PCA (Principal Component Analysis) shows how to reduce the dimensionality of a dataset by selecting the top 2 principal components. This suggests that one key finding is the importance of reducing dimensionality in machine learning models.\n\nAdditionally, the section on clustering algorithms mentions K-Means Clustering and DBSCAN as two types of clustering techniques. While this does not provide specific results or findings, it highlights the diversity of clustering methods available for unsupervised learning tasks.\n\nOverall, the key findings presented in the paper seem to be focused on feature engineering and dimensionality reduction techniques, with an emphasis on reducing complexity in machine learning models.'

In [20]:
print(result)

Based on the provided context, it appears that the document is a machine learning cheatsheet. The key findings or results presented in the paper seem to be related to feature engineering and dimensionality reduction techniques.

Specifically, the section on PCA (Principal Component Analysis) shows how to reduce the dimensionality of a dataset by selecting the top 2 principal components. This suggests that one key finding is the importance of reducing dimensionality in machine learning models.

Additionally, the section on clustering algorithms mentions K-Means Clustering and DBSCAN as two types of clustering techniques. While this does not provide specific results or findings, it highlights the diversity of clustering methods available for unsupervised learning tasks.

Overall, the key findings presented in the paper seem to be focused on feature engineering and dimensionality reduction techniques, with an emphasis on reducing complexity in machine learning models.
